In [17]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.prompts import PromptTemplate
from langsmith import Client
from langchain.tools import tool
import requests

In [18]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [19]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [20]:
search_tool = TavilySearch(max_results=2)

In [21]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [22]:
result = search_tool.invoke("Give me the latest news on AI")
result

{'query': 'Give me the latest news on AI',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://guides.library.georgetown.edu/ai/news',
   'title': 'Guides: Artificial Intelligence (Generative) Resources: Latest News on AI',
   'content': "U.A.E.-Based AI and Data Startup Expands Into Silicon Valley; CNTXT AI, which raised $60 million in June, has built Arabic-first AI models and is training robots This link opens in a new window_Rosenbush, Steven Wall Street Journal (Online) 19 Oct 2026Formats: Citation,Full text,Find Full Text @ GU,Find Full Text @ GU_ Oct 19, 2026 [...] Nvidia-Backed Startup doubleAI Launches Model to Pare Cost of Compute; The company, co-founded by Mobileye CEO Amnon Shashua, says it slashed one early customer's compute costs by 67%This link opens in a new window_Lin, Belle Wall Street Journal (Online) 19 Oct 2026Formats: Citation,Full text,Find Full Text @ GU,Find Full Text @ GU_ Oct 19, 2026 [...] DHS Orders DeepSeek Technol

In [23]:
# ==========================================
# LLM
# ==========================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)

In [ ]:
response = llm.invoke("Tell me a joke about AI")
response

In [25]:
# ==========================================
# PROMPT
# ==========================================

prompt = Client().pull_prompt(
    "hwchase17/react",
    include_model=False,
    dangerously_pull_public_prompt=True,
)
prompt = PromptTemplate.from_template(
    "Answer in the same language as the user's question. "
    "Keep proper nouns and technical terms unchanged when appropriate.\n\n"
    + prompt.template,
    template_format=prompt.template_format,
    metadata=prompt.metadata,
)

In [26]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template="Answer in the same language as the user's question. Keep proper nouns and technical terms unchanged when appropriate.\n\nAnswer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratch

In [27]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [28]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [29]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [30]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "Quelle est la capitale de la France ? "
        "Quelle est sa population ?"
        "Quel temps y fait-il ?"
    )
})




> Entering new AgentExecutor chain...
Je dois utiliser la fonction get_weather_data pour obtenir des informations sur la météo de la capitale de la France.
Action: get_weather_data
Action Input: "Paris"City: Paris
Temperature: 21°C
Weather: Sunny
Humidity: 33%Je dois maintenant utiliser la fonction tavily_search_results_json pour obtenir des informations sur la population de la capitale de la France.
Action: tavily_search_results_json
Action Input: "Population of Paris, France"[{'title': 'Paris - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/Paris', 'content': 'The population of the City of Paris was 2,102,650 in January 2023, down from 2,165,423 in January 2022. Between 2013 and 2023, the population fell by 122,919, or about five percent. The Mayor of Paris, Anne Hidalgo, declared that this illustrated the "de-densification" of the city, creating more green space and less crowding. Despite the drop, Paris remains the most densely populated city in Europe, with 252 residents per 

In [31]:
print(response["output"])

La capitale de la France est Paris. Sa population est d'environ 2,102,650 habitants. Il fait actuellement ensoleillé avec une température de 21°C.
